In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
OUTPUT_DIR = ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
route_records = []

for partition in ["train", "student_test"]:

    for trip_dir in sorted((DATA_DIR / partition).iterdir()):

        if not trip_dir.is_dir():
            continue

        trip_id = trip_dir.name

        arcs = pd.read_csv(
            trip_dir / f"{trip_id}.arcs",
            sep=r"\s+",
            header=None,
            names=["from_node", "to_node"]
        )

        route = pd.read_csv(
            trip_dir / f"{trip_id}.route",
            sep=r"\s+",
            header=None,
            names=["arc_id"]
        )

        # Validate route IDs
        route_ids = pd.to_numeric(
            route["arc_id"], errors="coerce"
        )

        integer_mask = (
            route_ids.notna()
            & (route_ids % 1 == 0)
        )

        in_range_mask = (
            integer_mask
            & route_ids.between(0, len(arcs) - 1)
        )

        valid_ids = route_ids[in_range_mask].astype(int).tolist()

        invalid_ids = int((~in_range_mask).sum())

        # Check consecutive route arc connectivity
        disconnected_pairs = 0
        checked_pairs = 0

        for i in range(len(valid_ids) - 1):

            current_arc = valid_ids[i]
            next_arc = valid_ids[i + 1]

            current_to = arcs.iloc[current_arc, 1]
            next_from = arcs.iloc[next_arc, 0]

            checked_pairs += 1

            if current_to != next_from:
                disconnected_pairs += 1

        route_records.append({

            "partition": partition,
            "trip_id": trip_id,

            "route_length": len(route),

            "valid_route_arcs": len(valid_ids),

            "invalid_route_ids": invalid_ids,

            "repeated_arc_ids": (
                len(valid_ids) - len(set(valid_ids))
            ),

            "checked_consecutive_pairs": checked_pairs,

            "disconnected_pairs": disconnected_pairs,

            "route_connectivity_percent": (
                (checked_pairs - disconnected_pairs)
                / checked_pairs * 100
                if checked_pairs else np.nan
            )
        })

route_summary = pd.DataFrame(route_records)

route_summary.to_csv(
    OUTPUT_DIR / "route_quality.csv",
    index=False
)

print("Route analysis completed.")
print("Trips analysed:", len(route_summary))

display(route_summary.head())

Route analysis completed.
Trips analysed: 90


,partition,trip_id,route_length,valid_route_arcs,invalid_route_ids,repeated_arc_ids,checked_consecutive_pairs,disconnected_pairs,route_connectivity_percent
0,train,00000001,525,525,0,0,524,0,100.000000
1,train,00000002,262,262,0,0,261,1,99.616858
2,train,00000003,79,79,0,0,78,0,100.000000
3,train,00000004,320,320,0,0,319,0,100.000000
4,train,00000005,105,105,0,0,104,0,100.000000


In [15]:
display(
    route_summary.groupby("partition").agg(
        total_route_arcs=("route_length", "sum"),
        invalid_route_ids=("invalid_route_ids", "sum"),
        repeated_arc_ids=("repeated_arc_ids", "sum"),
        disconnected_pairs=("disconnected_pairs", "sum"),
        median_route_length=("route_length", "median"),
        median_connectivity=("route_connectivity_percent", "median")
    )
)

,total_route_arcs,invalid_route_ids,repeated_arc_ids,disconnected_pairs,median_route_length,median_connectivity
partition,,,,,,
student_test,3477,0,255,1,250.0,100.0
train,31336,0,67,13,280.0,100.0


In [16]:
print("Routes with invalid arc IDs:")

display(
    route_summary[
        route_summary["invalid_route_ids"] > 0
    ]
)

print("Routes with disconnected consecutive arcs:")

display(
    route_summary[
        route_summary["disconnected_pairs"] > 0
    ]
)

Routes with invalid arc IDs:


,partition,trip_id,route_length,valid_route_arcs,invalid_route_ids,repeated_arc_ids,checked_consecutive_pairs,disconnected_pairs,route_connectivity_percent


Routes with disconnected consecutive arcs:


,partition,trip_id,route_length,valid_route_arcs,invalid_route_ids,repeated_arc_ids,checked_consecutive_pairs,disconnected_pairs,route_connectivity_percent
1,train,00000002,262,262,0,0,261,1,99.616858
14,train,00000019,545,545,0,0,544,3,99.448529
26,train,00000033,169,169,0,0,168,1,99.404762
33,train,00000041,340,340,0,0,339,1,99.705015
34,train,00000042,1440,1440,0,0,1439,1,99.930507
55,train,00000069,467,467,0,29,466,1,99.785408
60,train,00000075,108,108,0,2,107,2,98.130841
63,train,00000080,139,139,0,0,138,1,99.275362
69,train,00000086,671,671,0,2,670,1,99.850746
71,train,00000089,265,265,0,0,264,1,99.621212
